# 09g The THINGS-EEG2 release as a Kaggle dataset

**Kaggle:** CPU, Internet on. **Inputs:** none.

Downloads each of the 10 THINGS-EEG2 people from OSF once (Gifford et al. 2022, preprocessed release), checks shapes,
channels and times, and writes float32 copies in the same file format (every notebook loads the data as float32, so
results are unchanged; 5.5 GB instead of 11 GB). Files get flat names such as
`things-eeg2_sub-01_preprocessed_eeg_training.npy`. Also stores `image_metadata.npy` and a manifest with shapes, cast
errors and SHA-256. The folder is uploaded as the dataset `cross-subject-brain-encoding-eeg`, which later notebooks
attach.

In [ ]:
# Which notebook this is, printed and saved with the outputs.
NOTEBOOK_ID, RUN_MODE = '09g-eegdata', 'cpu-build'
import json, os
print("=" * 72 + f"\n  NOTEBOOK {NOTEBOOK_ID}  |  MODE {RUN_MODE}\n" + "=" * 72)
_out = "/kaggle/working/results/metrics" if os.path.exists("/kaggle") else "results/metrics"
os.makedirs(_out, exist_ok=True)
json.dump({"notebook_id": NOTEBOOK_ID, "mode": RUN_MODE, "expected": "not checked"}, open(os.path.join(_out, f"{NOTEBOOK_ID}_run_identity.json"), "w"))

In [ ]:
import os
import time
T0 = time.monotonic()
if not os.path.exists('/kaggle'):
    raise RuntimeError('Run on Kaggle')
import importlib
import json
import os
import re
import subprocess
import sys
from pathlib import Path

PACKAGE, DIST, REPO, REF = "cross_subject_encoding", "cross-subject-encoding", "Shakkak/Borrow-then-Personalize-EEG-fMRI-Brain-Encoding-with-Limited-Labels", "v1.0.0"  # REF: the release tag a GitHub install uses
IS_KAGGLE = os.path.exists("/kaggle")


def install_args() -> list[str]:
    if not IS_KAGGLE:
        return ["-e", str(Path.cwd().parent)]  # this checkout, editable
    # a wheel of this package attached as a Kaggle Dataset, if there is one
    wheels = sorted(Path("/kaggle/input").glob(f"**/{PACKAGE}-*.whl"))
    if wheels:
        return [str(wheels[-1])]
    # otherwise install the public repository
    return [f"git+https://github.com/{REPO}.git@{REF}"]


def code_version() -> str:
    from importlib.metadata import distribution
    try:
        info = json.loads(distribution(DIST).read_text("direct_url.json") or "{}")
    except Exception:
        info = {}
    if "vcs_info" in info:
        return info["vcs_info"].get("commit_id", "")[:7]
    root = Path(info["url"].removeprefix("file://")) if "url" in info else Path.cwd().parent
    root = root.parent if root.suffix == ".whl" else root  # COMMIT sits next to the wheel
    if (root / "COMMIT").exists():
        return (root / "COMMIT").read_text().strip()
    try:
        return subprocess.run(["git", "-C", str(root), "rev-parse", "--short", "HEAD"],
                              capture_output=True, text=True).stdout.strip() or "unknown"
    except FileNotFoundError:
        return "unknown"


try:
    pkg = importlib.import_module(PACKAGE)
except ImportError:
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *install_args()], capture_output=True, text=True)
    if r.returncode:  # hide any credential in a clone URL
        raise RuntimeError("pip install failed:\n" + re.sub(r"x-access-token:[^@]+@", "x-access-token:***@", r.stderr[-3000:]))
    if not IS_KAGGLE:  # an editable install is only picked up by a fresh interpreter, so point this one at src/
        sys.path.insert(0, str(Path.cwd().parent / "src"))
    importlib.invalidate_caches()
    pkg = importlib.import_module(PACKAGE)
print(f"Kaggle: {IS_KAGGLE} | {PACKAGE} {pkg.__version__} from {Path(pkg.__file__).parent} | code version: {code_version()}")

In [ ]:
FIXED = json.loads('{"people": ["sub-01", "sub-02", "sub-03", "sub-04", "sub-05", "sub-06", "sub-07", "sub-08", "sub-09", "sub-10"], "slug": "cross-subject-brain-encoding-eeg", "min_free_gb": 15}')
import shutil
import numpy as np

from cross_subject_encoding.data import things_eeg2 as te
from cross_subject_encoding.utils import config as C
from cross_subject_encoding.utils.eeg1_pilot import sha256_file
from cross_subject_encoding.utils.kaggle_export import export_folder

OUT, WORK = Path("/kaggle/tmp/eeg_dataset"), Path("/kaggle/tmp/eeg_download")
OUT.mkdir(parents=True, exist_ok=True)
free_gb = shutil.disk_usage(OUT).free / 1e9
print(f"free disk under /kaggle/tmp: {free_gb:.1f} GB")
if free_gb < FIXED["min_free_gb"]:
    raise RuntimeError(f"need about {FIXED['min_free_gb']} GB free (5.5 GB dataset plus one person's 2 GB download)")
manifest_path = OUT / "09g_eeg_manifest.json"
manifest = json.loads(manifest_path.read_text()) if manifest_path.exists() else {
    "source": "THINGS-EEG2 preprocessed release on OSF (Gifford et al. 2022); float32 copies of the float64 files",
    "format": "np.save of a dict: preprocessed_eeg_data (float32), ch_names, times; read with things_eeg2.load_part",
    "people": {}}
reference = None
for s in FIXED["people"]:
    t = time.monotonic()
    targets = {part: OUT / te.FLAT.format(subject=s, part=part) for part in ("training", "test")}
    if s in manifest["people"] and all(p.exists() for p in targets.values()):
        print(s, "already written")
        continue
    te.download_subject(s, WORK)
    info = {}
    for part, dst in targets.items():
        info[part] = te.write_float32(te.subject_file(WORK, s, part), dst)
        loaded = te.load_part(dst)
        problems = te.check_part(loaded, part)
        if problems:
            raise ValueError(f"{s}: {problems}")
        layout = (loaded["ch_names"], loaded["times"].round(4).tolist())
        reference = reference or layout
        if layout != reference:
            raise ValueError(f"{s}/{part}: channels or times differ from the first person's")
        info[part]["sha256"] = sha256_file(dst)
        del loaded
    shutil.rmtree(WORK / s, ignore_errors=True)
    manifest["people"][s] = info
    manifest_path.write_text(json.dumps(manifest, indent=1))
    print(s, f"{time.monotonic() - t:.0f} s | cast error {max(v['max_cast_error'] for v in info.values()):.2e} |",
          f"{(time.monotonic() - T0) / 60:.1f} min")
if not (OUT / "image_metadata.npy").exists():
    te.fetch(te.OSF.format(C.EEG_OSF_IMAGE_METADATA), OUT / "image_metadata.npy")
manifest["image_metadata_sha256"] = sha256_file(OUT / "image_metadata.npy")
manifest["channels"], manifest["times"] = reference if reference else (manifest.get("channels"), manifest.get("times"))
manifest["total_gb"] = sum(p.stat().st_size for p in OUT.glob("*.npy")) / 1e9
manifest_path.write_text(json.dumps(manifest, indent=1))
shutil.copy2(manifest_path, C.METRICS_DIR / manifest_path.name)
assert len(manifest["people"]) == len(FIXED["people"]), "not every person was written"
print(f"{len(manifest['people'])} people, {manifest['total_gb']:.2f} GB |", f"{(time.monotonic() - T0) / 60:.1f} min")

In [ ]:
url = export_folder(FIXED["slug"], "XSUB THINGS-EEG2 preprocessed EEG (float32)", OUT, message="THINGS-EEG2, 10 people, float32")
print(url, "|", f"{(time.monotonic() - T0) / 60:.1f} min")